# 1. Título: 

### Dados Consolidados de Movimentação de Gás Natural em Gasodutos de Transporte

# 2. Membros (nome e número de matrícula): 

### Igor Braga de Lima - 2021019351
### Arthur Felipe Reis Souza - 2021013884
### Raphael Henrique Braga Leivas - 2020028101

# 3. Descrição dos dados (qual a URL? qual o domínio? como os dados foram processados?)

URL: https://dados.gov.br/dados/conjuntos-dados/dados-consolidados-de-movimentacao-de-gas-natural-em-gasodutos-de-transporte
Domínio: Agência Nacional de Petróleo, Gás Natural e Biocombustíveis

Inicialmente completou-se as datas faltantes, seguindo a ordem cronologica da amostragem. Em seguida 

# 4. Diagrama ER

![title](img/diagrama-er.png)

# 5. Diagrama relacional

![title](img/modelo-relacional.png)

# 6. Consultas

In [1]:
import pandas as pd
import sqlite3

In [39]:
# Conectando ao banco de dados SQLite
conn = sqlite3.connect('gas_data.db')

# Listando as tabelas no banco de dados
tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables_df = pd.read_sql_query(tables_query, conn)

print("As tabelas disponíveis no banco de dados são:")
tables_df

# Loop em todas as tabelas para exibir as primeiras linhas
# for table_name in tables_df['name']:
#     print(f"\n📄 Primeiras linhas da tabela: {table_name}")
#     df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 5;", conn)
#     print(df)

As tabelas disponíveis no banco de dados são:


,name
0,Operador
1,Carregador
2,Instalacao_Transporte
3,Instalacao_Gasoduto
4,Contrato
5,Medicao
6,Variavel


## 6.1 Duas consultas envolvendo seleção e projeção

### 6.1.1 Consulta 1

In [40]:
print("--- Consulta 1: Instalações de Gasoduto em Macaé ---")
conn = sqlite3.connect('gas_data.db')

query1 = """
SELECT
    codigo_instalacao_gasoduto,
    nome_instalacao_gasoduto
FROM
    Instalacao_Gasoduto
WHERE
    municipio = 'Macaé';
"""

df1 = pd.read_sql_query(query1, conn)

conn.close()

df1


--- Consulta 1: Instalações de Gasoduto em Macaé ---


,codigo_instalacao_gasoduto,nome_instalacao_gasoduto
0,220981,Interconexão TECAB (TECAB >> GASCAV)
1,220988,Interconexão TECAB (GASCAV >> TECAB)
2,222296,Interconexão TECAB (TECAB >> GASDUC III)
3,222298,Interconexão TECAB (GASDUC III >> TECAB)
4,222300,UTE Norte Fluminense
5,222301,UTE Mário Lago
6,622774,INTERCONEXÃO CABIÚNAS (GASCAV >> GASDUC)
7,622776,INTERCONEXÃO CABIÚNAS (GASDUC >> GASCAV)
8,622844,INTERCONEXÃO CABIÚNAS (GASDUC >> GASCAV)
9,622845,INTERCONEXÃO CABIÚNAS (GASCAV >> GASDUC)


### 6.1.2 Consulta 2

In [ ]:
print("--- Consulta 2: Carregadores específicos - Petrobras ou Shell ---")
conn = sqlite3.connect('gas_data.db')

query2 = """
SELECT
    codigo_carregador,
    nome_carregador
FROM
    Carregador
WHERE
    nome_carregador = 'Petróleo Brasileiro S.A. - PETROBRAS' OR nome_carregador = 'Shell Energy do Brasil Gás Ltda';
"""

df2 = pd.read_sql_query(query2, conn)

conn.close()

df2

--- Consulta 2: Nome e Município de Instalações 'Ponto de Recebimento' no RJ ---


,codigo_carregador,nome_carregador
0,33000167,Petróleo Brasileiro S.A. - PETROBRAS
1,9600150046,Shell Energy do Brasil Gás Ltda


## 6.2 Três consultas envolvendo junção de duas relações

### 6.2.1 Consulta 3

In [41]:
print("--- Consulta 3: Listar todos os contratos com seus respectivos dados de medição. ---")

conn = sqlite3.connect('gas_data.db')

query3 = """
SELECT DISTINCT
    C.nome_contrato,
    M.data_medicao,
    M.valor,
    V.unidade_medida
FROM
    Contrato AS C
INNER JOIN
    Instalacao_Transporte as IT ON IT.codigo_instalacao_transporte = C.codigo_instalacao_transporte
INNER JOIN
    Instalacao_Gasoduto as IG ON IG.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN 
    Medicao as M ON M.codigo_instalacao_gasoduto = IG.codigo_instalacao_gasoduto
INNER JOIN
    Variavel as V ON M.nome_variavel = V.nome_variavel
"""

df3 = pd.read_sql_query(query3, conn)

conn.close()

df3


--- Consulta 3: Listar todos os contratos com seus respectivos dados de medição. ---


,nome_contrato,data_medicao,valor,unidade_medida
0,Gasocidente do Mato Grosso Ltda x AMBAR Energi...,2025-03-01 00:00:00,0.00,mil m³
1,Gasocidente do Mato Grosso Ltda x Companhia Ma...,2025-03-01 00:00:00,0.00,mil m³
2,Gasocidente do Mato Grosso Ltda x AMBAR Energi...,2025-03-01 00:00:00,0.00,%
3,Gasocidente do Mato Grosso Ltda x Companhia Ma...,2025-03-01 00:00:00,0.00,%
4,Gasocidente do Mato Grosso Ltda x AMBAR Energi...,2025-03-01 00:00:00,82.30,kgf/cm²
...,...,...,...,...
171011,SHELL,2025-03-31 00:00:00,60.97,kgf/cm²
171012,Fronteira (Argentina) - Uruguaiana (Trecho 1),2025-03-31 00:00:00,0.00,mil m³
171013,Fronteira (Argentina) - Uruguaiana (Trecho 1),2025-03-31 00:00:00,100.00,%
171014,Fronteira (Argentina) - Uruguaiana (Trecho 1),2025-03-31 00:00:00,38.50,kgf/cm²


### 6.2.2 Consulta 4

In [43]:
print("--- Consulta 4: Calcular o valor total medido para cada contrato. ---")
conn = sqlite3.connect('gas_data.db')

query4 = """
SELECT
    C.nome_contrato,
    SUM(M.valor) AS ValorTotalMedido
FROM
    Contrato AS C
INNER JOIN
    Instalacao_Transporte as IT ON IT.codigo_instalacao_transporte = C.codigo_instalacao_transporte
INNER JOIN
    Instalacao_Gasoduto as IG ON IG.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN 
    Medicao as M ON M.codigo_instalacao_gasoduto = IG.codigo_instalacao_gasoduto
GROUP BY
    C.nome_contrato;
"""

df4 = pd.read_sql_query(query4, conn)

conn.close()

df4

--- Consulta 4: Calcular o valor total medido para cada contrato. ---


,nome_contrato,ValorTotalMedido
0,3R PETROLEUM OFFSHORE,180423.57
1,3R POTIGUAR,170943.72
2,3R RNCE,78441.30
3,ALGAS,32266.20
4,BRAVA,6867434.18
5,CEGAS,78441.30
6,COMPAGAS,1605773.01
7,COPERGAS,133515.35
8,CP0119SC1-S2025-PET,1605773.01
9,CP0120SC2-S2025-PET,1605773.01


### 6.2.3 Consulta 5

In [44]:
print("--- Consulta 5: Encontrar contratos que possuem medições para uma instalação de transporte específica. ---")
conn = sqlite3.connect('gas_data.db')

query5 = """
SELECT DISTINCT
    C.nome_contrato,
    C.codigo_instalacao_transporte
FROM
    Contrato AS C
INNER JOIN
    Instalacao_Transporte as IT ON IT.codigo_instalacao_transporte = C.codigo_instalacao_transporte
INNER JOIN
    Instalacao_Gasoduto as IG ON IG.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN 
    Medicao as M ON M.codigo_instalacao_gasoduto = IG.codigo_instalacao_gasoduto
WHERE
    C.codigo_instalacao_transporte = '505444'; -- Exemplo para o contrato 'Fronteira (Argentina) - Uruguaiana'
"""

df5 = pd.read_sql_query(query5, conn)

conn.close()

df5


--- Consulta 5: Encontrar contratos que possuem medições para uma instalação de transporte específica. ---


,nome_contrato,codigo_instalacao_transporte
0,Fronteira (Argentina) - Uruguaiana (Trecho 1),505444


## 6.3 Três consultas envolvendo junção de três ou mais relações

### 6.3.1 Consulta 6

In [45]:

print("--- Consulta 6: Encontrar carregadores da instalação de gasoduto Corumbá ---")
conn = sqlite3.connect('gas_data.db')

query6 = """
SELECT DISTINCT CA.codigo_carregador, CA.nome_carregador
FROM
    Contrato AS C
INNER JOIN 
    Instalacao_Transporte as IT
ON
    C.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN
    Instalacao_Gasoduto as IG
ON
    IG.codigo_instalacao_transporte = IT.codigo_instalacao_transporte
INNER JOIN
    Carregador as CA
ON
    CA.codigo_carregador = C.codigo_carregador
WHERE
    IG.nome_instalacao_gasoduto = "Corumbá"
"""

df6 = pd.read_sql_query(query6, conn)

conn.close()

df6

--- Consulta 6: Encontrar carregadores da instalação de gasoduto Corumbá ---


,codigo_carregador,nome_carregador
0,535681,COMPAGAS
1,16974249,GALP
2,33000167,Petróleo Brasileiro S.A. - PETROBRAS
3,72300122,SCGAS
4,86864543,SULGAS
5,3002741679,MSGAS
6,3004423567,ENEVA
7,3034456148,YPFB
8,3056123140,MTX
9,7719046324,EDGE


### 6.3.2 Consulta 7

### 6.3.3 Consulta 8

## 6.4 Duas consultas envolvendo agregação sobre junção de duas ou mais relações

### 6.4.1 Consulta 9

### 6.4.2 Consulta 10

# 7. Autoavaliação dos membros